In [1]:
import torch
from geister_game import GeisterGame
from train import run_geister_cnn_training, run_geister_cqcnn_training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# CNNモデルの学習・評価
run_geister_cnn_training(episodes=300)


In [ ]:
# CQCNNモデルの学習・評価
run_geister_cqcnn_training(episodes=100, n_qbits=6, cnn_out_feat=6)


In [2]:
import json
import torch
import pennylane as qml
from model_cqcnn import CNN_QNN_CNN_Geister

# 設定ファイルの読み込み
with open("models_geister_agentA/agentA_cqcnn_config.json", "r") as f:
    config = json.load(f)
# 正しい PennyLane デバイスを構築
dev = qml.device(config.get("dev_type", "default.qubit"), wires=config["n_qubits_qnn"])


# モデルの構築
model = CNN_QNN_CNN_Geister(
    dev=dev,  # ← 修正ポイント
    n_qubits_qnn=config["n_qubits_qnn"],
    exp_or_prob=config["exp_or_prob"],
    embedding_type=config["embedding_type"],
    ansatz_type=config["ansatz_type"],
    feature_map_reps=config["feature_map_reps"],
    ansatz_reps=config["ansatz_reps"],
    input_channels_cnn=config["input_channels_cnn"],
    board_size_cnn=config["board_size_cnn"],
    cnn_fc_out_features=config["cnn_fc_out_features"],
    qnn_fc_out_features=config["qnn_fc_out_features"]
)

# 重みの読み込み
model.load_state_dict(torch.load("models_geister_agentA/agentA_cqcnn_eps500.pth", map_location=device))
model.eval()  # 評価モードにする

CNN_QNN_CNN_Geister(
  (cnn_feature_extractor): Sequential(
    (0): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (fc_to_qnn): Linear(in_features=1152, out_features=6, bias=True)
  (fc_from_qnn): Linear(in_features=6, out_features=36, bias=True)
)

In [15]:
# ゲームインスタンスとエージェントを作成
from train import Env_Geister
from train import AgentFactory
game = GeisterGame(board_size=6)
# 設定をファイルから読み込む
with open('models_geister_agentA/agentA_cqcnn_config.json', 'r') as f:
    configA = json.load(f)

with open('models_geister_agentA/agentA_cqcnn_config.json', 'r') as f:
    configB = json.load(f)
agentA = AgentFactory.create_cqc_agent("A", game, configA, 'models_geister_agentA/agentA_cqcnn_eps100.pth')
agentB = AgentFactory.create_cqc_agent("B", game, configB, 'models_geister_agentA/agentA_cqcnn_eps3000.pth')

# 環境インスタンス作成
env = Env_Geister(agentA, agentB, game)

# 対戦実行
winner, moves_log, final_board = env.play_one_game_with_log()
print("Winner:", winner)
print("Moves Log:", moves_log)

Winner: B
Moves Log: [{'turn': 1, 'player': 'A', 'action': ((4, 2), (3, 2))}, {'turn': 2, 'player': 'B', 'action': ((0, 1), (0, 0))}, {'turn': 3, 'player': 'A', 'action': ((3, 2), (2, 2))}, {'turn': 4, 'player': 'B', 'action': ((0, 0), (1, 0))}, {'turn': 5, 'player': 'A', 'action': ((4, 1), (4, 0))}, {'turn': 6, 'player': 'B', 'action': ((0, 2), (0, 1))}, {'turn': 7, 'player': 'A', 'action': ((4, 4), (3, 4))}, {'turn': 8, 'player': 'B', 'action': ((1, 0), (2, 0))}, {'turn': 9, 'player': 'A', 'action': ((2, 2), (2, 1))}, {'turn': 10, 'player': 'B', 'action': ((1, 4), (1, 5))}, {'turn': 11, 'player': 'A', 'action': ((5, 1), (4, 1))}, {'turn': 12, 'player': 'B', 'action': ((1, 5), (0, 5))}, {'turn': 13, 'player': 'A', 'action': ((3, 4), (3, 3))}, {'turn': 14, 'player': 'B', 'action': ((0, 5), (1, 5))}, {'turn': 15, 'player': 'A', 'action': ((3, 3), (3, 2))}, {'turn': 16, 'player': 'B', 'action': ((0, 1), (0, 0))}, {'turn': 17, 'player': 'A', 'action': ((4, 3), (3, 3))}, {'turn': 18, 'play

In [ ]:
print("\n--- Evaluating Trained CQCNN Agent A vs Random ---")
agentA.eval_mode_on()
agentA.epsilon = 0.0 # 評価時はランダム性なし
agentB.eval_mode_on()
agentB.epsilon = 0.0 # 評価時はランダム性なし
eval_env_Q_vs_env_Q = env
eval_env_Q_vs_env_Q.start_training(episodes=100, visualize_interval=0, train_agents=False)

In [3]:
from train import run_geister_cqcnn_training
from geister_game import GeisterGame

# 例: 4x4ボードで学習（かつゴーストは4つ）
custom_game = GeisterGame(board_size=4, num_ghosts_per_player=2)

# 外から渡す
run_geister_cqcnn_training(episodes=3000, n_qbits=4, cnn_out_feat=4, game_instance=custom_game)


--- Training CQCNN Agent (Qubits: 4, CNN->QNN Feat: 4) ---
--- Episode 100/3000 ---
Agent A Wins: 48 (48.00%), Agent B Wins: 52 (52.00%), Draws: 0 (0.00%)
Avg Loss A (last 500): nan, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4902973499203101
Agent B Epsilon: 0.4902973499203101
--- Episode 200/3000 ---
Agent A Wins: 88 (44.00%), Agent B Wins: 112 (56.00%), Draws: 0 (0.00%)
Avg Loss A (last 500): nan, Avg Loss B (last 500): nan
Agent A Epsilon: 0.48078682518463833
Agent B Epsilon: 0.48078682518463833
--- Episode 300/3000 ---
Agent A Wins: 138 (46.00%), Agent B Wins: 162 (54.00%), Draws: 0 (0.00%)
Avg Loss A (last 500): nan, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4714646214562819
Agent B Epsilon: 0.4714646214562819
--- Episode 400/3000 ---
Agent A Wins: 189 (47.25%), Agent B Wins: 211 (52.75%), Draws: 0 (0.00%)
Avg Loss A (last 500): nan, Avg Loss B (last 500): nan
Agent A Epsilon: 0.4623270097294515
Agent B Epsilon: 0.4623270097294515
--- Episode 500/3000 ---
Agent A Wins: 235